Recall that the coboundary operator is

$$\partial: C^k(\mathfrak{m},\mathfrak{g})\to C^{k+1}(\mathfrak{m},\mathfrak{g})$$

and is defined by

$$\partial\phi(\alpha_0,\ldots,\alpha_{k}) = \sum_{i=0}^{k}(-1)^i\big[\alpha_i,\phi(\alpha_0,\ldots,\hat\alpha_i,\ldots, \alpha_{k})\big]$$
$$+ \sum_{i<j}(-1)^{i+j}\phi\big([\alpha_i,\alpha_j],\alpha_0,\ldots, \hat \alpha_i,\ldots,\hat\alpha_j,\ldots,\alpha_{k}\big).$$

For a basis element $\phi = X_1^*\wedge\cdots\wedge X_k^*\otimes Y$, we have
$$
    \partial\phi = \sum_{a\not\in\{1,\ldots,k\}} X_a^*\wedge X_1^*\wedge\cdots \wedge X_k^*\otimes[X_a,Y]
    \\
    +\sum_{a<b\not\in\{1,\ldots, k\}}\sum_{i\in \{1,\ldots, k\}}\Big(-X_i^* [X_a,X_i]\Big)X_a^*\wedge X_b^*\wedge X_1^*\wedge\cdots\wedge\widehat {X_i^*}\wedge\cdots X_k^*\otimes Y
$$


We consider the inner product on the Tanaka symbol with orthonormal basis $(Y,H,E,X,\varepsilon_1,\ldots,\varepsilon_{2m},\eta)$ and lengths

$$|Y|^2=|X|^2=1,|H|^2=|E|^2=2,|\varepsilon_i|^2=\frac{(i-1)!}{(2m-i)!}, |\eta|^2=1$$

along with the innerproduct induced on tensor spaces. In particular, 

$$|A^*\wedge B^*\otimes C|^2 = \frac{|C|^2}{|A|^2|B|^2}$$

In [1]:
%run T_symb.ipynb
%run helpers.ipynb
import copy
from sympy import *
from itertools import combinations
import time
from sympy.matrices.sparsetools import _doktocsr

In [2]:
def key_from_neg(c_key):
    '''arg: c_key a tuple of strings like ('H','X','e1')
       returns: True if c_key is from the complex C(m,g), False otherwise.'''
    # # To do: Rewrite this to be more general
    for i in range(len(c_key)-1):
        A=c_key[i]
        if A in ('H','E','Y'): return False
    return True

In [3]:
class Tensor_alg(object):
    def __init__(self,T_symb_obj):
        self.alg=T_symb_obj
        self.basis_cache={}
        self.dwi_dicts={} # Keys: str_reps of basis elts; Values:(deg,wght,index)
        self.childcls=None
        
    def elt(self,vd):
        return self.childcls(self,vd)

    def elt_from_cd(self,cd={}):
        return self.childcls.from_cd(self,cd)
    
    def basis(self,deg,wght=None):
        """Returns a basis for (deg, wght) as a list, or if wght==None, returns 
        basis for deg as a dict of wghts."""
        if deg<0: return []
        self.init_basis(deg)
        if wght==None:
            return self.basis_cache[deg]
        if deg not in self.basis_cache or wght not in self.basis_cache[deg]: return []
        return self.basis_cache[deg][wght]
    
    def basis_strs(self,deg,wght=None):
        b=self.basis(deg,wght)
        if wght==None:
            r={}
            for k in b:
                r[k]=[tuple([str(c) for c in A.components]) for A in b[k]]
            return r
        return [tuple([str(c) for c in A.components]) for A in b]
        
    def tuple_deg(self,t):
        NotImplemented
    
    def tuple_wght(self,t):
        NotImplemented
    
    def sort_tuple(self,t):
        NotImplemented
        
    def cd_to_vd(self,cd={}):
        """Converts a coeff dict to a vector dict in the basis of self
        """
        r={}
        for A in cd:
            d=self.tuple_deg(A)
            w=self.tuple_wght(A)
            A1,s=self.sort_tuple(A)
            if d not in r: r[d]={}
            if w not in r[d]: r[d][w]=SparseMatrix(zeros(len(self.basis(d,w)),1))
            v=SparseMatrix(zeros(len(self.basis(d,w)),1))
            v[self.dwi_dicts[A1][2]]=s*cd[A]
            r[d][w]=r[d][w]+v
        return r
    
    def Q(self,d,w):
        """Returns the inner product matrix induces by that on the 
        base algebra for degree d and weight w
        INPUTS:
        * 'd' - a degree
        * 'w' - a weight
        """
        if self.alg.Q==None: print('Inner product on base algebra not initialized')
        return SparseMatrix(diag(*[A.length for A in self.basis(d,w)]))
    
    def iprod(self,elt1,elt2):
        """Returns the inner product of elt1 and elt2
        INPUTS:
        * 'elt1', 'elt2' - elements of self
        """
        return elt1.iprod(elt2)
    
    

In [4]:
class Tensor_alg_elt(object):
    def __init__(self,parent,vd={}):
        """INPUTS:
        * 'parent' - A tensor algebra
        * 'vd' - a dict of dicts of vectors, keyed by degree then weight"""
        self.parent=parent
        self.vd=vd
    
    @classmethod
    def from_cd(cls,parent,cd={}):
        NotImplemented
    
    def __str__(self):
        return str_from_vd(self.vd,self.parent.basis)
    
    def __repr__(self):
        return self.__str__()
    
    def __eq__(self,other):
        z=self-other
        return z.is_zero()
    
    def is_zero(self):
        for d in self.vd:
            for w in self.vd[d]:
                if simplify(self.vd[d][w])!=zeros(*shape(self.vd[d][w])): return False
        return True
    
    def __neg__(self):
        nvd={d:copy.copy(self.vd[d]) for d in self.vd}
        for d in nvd:
            for w in nvd[d]:
                nvd[d][w]=-nvd[d][w]
        return self.parent.elt(nvd)
        
    def __add__(self,other):
        nvd={}
        for d in set(self.vd.keys()).union(other.vd.keys()):
            nvd[d]={}
            if d in self.vd:
                if d in other.vd: 
                    for w in set(self.vd[d].keys()).union(set(other.vd[d].keys())):
                        if w in self.vd[d]:
                            if d not in nvd: nvd[d]={}
                            nvd[d][w]=copy.copy(self.vd[d][w])
                            if w in other.vd[d]:
                                nvd[d][w]=nvd[d][w]+other.vd[d][w]
                        else: nvd[d][w]=other.vd[d][w]
                else: nvd[d]=copy.copy(self.vd[d])
            else: nvd[d]=copy.copy(other.vd[d])
        return self.parent.elt(nvd)
    
    def __radd__(self,other):
        return self+other
    
    def __sub__(self,other):
        return self+(-other)
    
    def __mul__(self,k):
        nvd={}
        for d in self.vd:
            nvd[d]={}
            for w in self.vd[d]:
                nvd[d][w]=k*self.vd[d][w]
        return self.parent.elt(nvd)
    
    def __rmul__(self,other):
        return self*other
    
    def iprod(self,other):
        if not hasattr(other,'parent'): raise ValueError('arg of iprod must have parent')
        if self.parent!=other.parent: raise invalid_parent_exception('args of iprod must have the same parent')
        r=0
        for d in set(self.vd.keys()).intersection(set(other.vd.keys())):
            for w in set(self.vd[d].keys()).intersection(set(other.vd[d].keys())):
                r+=(self.vd[d][w].transpose()*self.parent.Q(d,w)*other.vd[d][w])[0]
        return r
    
    def subs(self,subs_dict):
        nvd={d:copy.copy(self.vd[d]) for d in self.vd}
        for d in nvd:
            for w in nvd[d]:
                nvd[d][w]=nvd[d][w].subs(subs_dict)
        return self.parent.elt(nvd)
    
    def large_subs(self,subs_dict):
        # # To Do
        # cd=copy.copy(self.coeff_dict)
        # for k in cd:
        #     IF=Indexed_factors(cd[k])
        #     NS={}
        #     for A in IF:
        #         if A in subs_dict: NS[A]=subs_dict[A]
        #     cd[k]=cd[k].subs(NS)
        # return self.parent.cochain(cd)
        NotImplemented
        
    def update_add(self,d,w,i,c):
        """Adds c times the specified basis element to self
        INPUTS:
        * 'd' = degree
        * 'w' = weight
        * 'i' = index
        * 'c' = coefficient
        """
        if not d in self.vd: self.vd[d]={}
        if not w in self.vd[d]: self.vd[d][w]=SparseMatrix(zeros(len(self.parent.basis(d,w)),1))
        self.vd[d][w][i]=self.vd[d][w][i]+c
        
    def update_add_vec(self,d,w,v):
        """Adds vector v to the specified degree and weight of self
        INPUTS:
        * 'd' - degree
        * 'w' - weight
        * 'v' - vector
        """
        if not d in self.vd: self.vd[d]={}
        if not w in self.vd[d]: self.vd[d][w]=SparseMatrix(v)
        self.vd[d][w]=SparseMatrix(self.vd[d][w]+v)
        
    def update_add_dict(self,ovd):
        """Adds the vector dict ovd to self
        INPUTS:
        * 'ovd' - a vector dict
        """
        for d in ovd:
            for w in ovd[d]: self.update_add_vec(d,w,ovd[d][w])

In [6]:
class cochain_complex(Tensor_alg):
    def __init__(self,T_symb_obj):
        T_symb_obj.cochain_complex=self
        Tensor_alg.__init__(self,T_symb_obj)
        self.alg=T_symb_obj
        self.ext_alg=T_symb_obj.ext_alg
        self.childcls=cochain
        self.cb_mat_cache={}
        self.subspace_cache={}
    
    def tuple_wght(self,t):
        ''' Returns the weight of t
        INPUTS:
        * 't' - a tuple representing an element of self
        '''
        r=0
        for i in range(len(t)-1):
            j=self.alg.basis_strs.index(t[i])
            r=r-self.alg.wght_list[j] # This is the exterior algebra of m_dual
        r+=self.alg.wght_list[self.alg.basis_strs.index(t[-1])]
        return r
        
    def tuple_deg(self,t):
        ''' Returns the degree of t
        INPUTS:
        * 't' - a tuple representing an element of self
        '''
        return len(t)-1
    
    def dwi(self,basis_str_tuple):
        """Returns the degree, weight, and index of basis_tuple
        Inputs:
        * 'basis_tuple' -- a tuple of basis strings representing a basic cochain
        """
        if basis_str_tuple not in self.dwi_dicts: self.init_basis(len(basis_str_tuple)-1)
        return self.dwi_dicts[basis_str_tuple]
    
    def sort_tuple(self,t):
        """Returns a (cochain)sorting of t and the sign of the corresponding permutation, as a tuple
        INPUTS:
        * 't' - a tuple of basis_strs
        """
        temp=t[0:-1]
        if len(temp)!=len(set(temp)): return (None, 0)
        r,s=sort_basis_tuple(temp,self.alg.basis_strs)
        return (tuple(list(r)+[t[-1]]),s)
        
    def init_basis(self,deg):
        """Sets value of deg in basis_cache and adds to basis_dicts"""
        if deg in self.basis_cache: return None
        deg_subsets=[]
        for A in combinations([str(A) for A in self.alg.m_basis],deg):
            for B in self.alg.basis_strs:
                deg_subsets.append(tuple(list(A)+[B]))
        self.basis_cache[deg]={}
        wght_ct={}
        for i in range(len(deg_subsets)):
            # count the number of elements of deg d and wght w
            # and set the deg, wght, and index of each elt in dwi_dicts
            A=deg_subsets[i]
            w=self.tuple_wght(A)
            if not w in wght_ct: 
                j=0
                wght_ct[w]=1
            else: 
                j=wght_ct[w]
                wght_ct[w]+=1
            self.dwi_dicts[A]=(deg,w,j)
        for A in deg_subsets:
            d,w,i=self.dwi_dicts[A]
            vec=SparseMatrix(zeros(wght_ct[w],1))
            vec[i]=1
            vd={d:{w:vec}}
            b_elt=cochain_basis_elt(self,d,w,vd,A)
            if i==0:self.basis_cache[deg][w]=[b_elt]
            else: self.basis_cache[deg][w].append(b_elt)
                
    def cb_mat(self,d,w):
        """Returns the coboundary matrix which operates on C^d_w
        INPUTS:
        * 'd' - degree
        * 'w' - weight
        """
        if d in self.cb_mat_cache and w in self.cb_mat_cache[d]: return self.cb_mat_cache[d][w]
        if self.basis(d+1,w)==[]: self.cb_mat_cache[(d,w)]=zeros(0,len(self.basis(d,w)))
        elif self.basis(d,w)==[]: self.cb_mat_cache[(d,w)]=zeros(len(self.basis(d+1,w)),0)
        else: self.cb_mat_cache[(d,w)]=SparseMatrix([list(c.cb_vec()) for c in self.basis(d,w)]).transpose()
        return self.cb_mat_cache[(d,w)]
    
    def cb(self, c):
        '''Returns the coboundary map of C(m,g) applied to c
        INPUTS:
        * 'c' - a cochain with self as parent'''
        return c.cb()
    
    def subspace_proj(self,c,subspace):
        """Returns the projection of c onto subspace
        INPUTS:
        * 'subspace' - among 'closed', 'coclosed', 'exact', 'coexact', and 'harmonic'
        * 'c' - a cochain from self
        """
        r={}
        for d in c.vd:
            r[d]={}
            for w in c.vd[d]:
                r[d][w]=ortho_proj(c.vd[d][w],self.subspace_basis(subspace,d,w),self.Q(d,w))
        return self.elt(r)
        
    def subspace_basis(self,subspace,d,w):
        """Returns a basis for the subspace in degree d and weight w
        INPUTS:
        * 'subspace' - among 'closed', 'coclosed', 'exact', 'coexact', and 'harmonic'
        * 'd' - a degree
        * 'w' - a weight
        """
        if len(self.basis(d,w))==0: return Matrix([])
        if (subspace,d,w) in self.subspace_cache: return self.subspace_cache[(subspace,d,w)]
        
        # Should I be caching here? It may be a waste of memory...
        if subspace=='closed':
            col_list=self.cb_mat(d,w).nullspace()
        if subspace=='coclosed':
            col_list=Mat_adjoint(self.cb_mat(d-1,w),self.Q(d-1,w),self.Q(d,w)).nullspace()
        if subspace=='exact':
            col_list=self.cb_mat(d-1,w).columnspace()
        if subspace=='coexact':
            col_list=Mat_adjoint(self.cb_mat(d,w),self.Q(d,w),self.Q(d+1,w)).columnspace()
        if subspace=='harmonic':
            N=(self.cb_mat(d,w)*self.subspace_basis('coexact',d,w)).nullspace()
            col_list=[self.subspace_basis('coexact',d,w)*v for v in N]
        if len(col_list)==0: self.subspace_cache[(subspace,d,w)]=zeros(len(self.basis(d,w)),0)
        else: self.subspace_cache[(subspace,d,w)]=Matrix([list(A) for A in col_list]).transpose()
        return self.subspace_cache[(subspace,d,w)]  
    
    def cb_preim_elt(self,c):
        """Returns a cochain which maps to c under the coboundary.
        If c is not exact, returns None.
        INPUTS:
        * 'c' - an exact cochain
        """
        r={}
        for d in c.vd:
            for w in c.vd[d]:
                t=new_Mat_preim_elt(self.cb_mat(d-1,w),c.vd[d][w])
                if t==None: return None
                if d-1 not in r:
                    r[d-1]={}
                if w not in r[d-1]: r[d-1][w]=t
                else: r[d-1][w]=r[d-1][w]+t
        return self.elt(r)
    
    def curv_dict_to_cochain(self,c):
        """Returns a cochain from self representing the structure function or curvature c
        INPUTS:
        * 'c' -- a dictionary repping a structure function {(i,j): vec rep of [Xi,Xj]}
        """
        r={}
        for i in range(len(self.alg.basis)-len(self.alg.m_basis),len(self.alg.basis)):
            for j in range(i+1,len(self.alg.basis)):
                for k in range(len(self.alg.basis)):
                    if c[(i,j)][k]!=0:
                        r[(self.alg.basis_strs[i],self.alg.basis_strs[j],self.alg.basis_strs[k])]=c[(i,j)][k]
        return self.elt_from_cd(r)

    def curv_cochain_to_dict(self,c):
        """Returns a dict of form {(i,j): v such that F*v=[Fi,Fj]-[Xi,Xj]} representing the cochain
        * 'c' -- a dictionary repping a structure function {(i,j): vec rep of [Xi,Xj]}
        """
        r={}
        for i in range(len(self.alg.basis)-len(self.alg.m_basis),len(self.alg.basis)):
            for j in range(i+1,len(self.alg.basis)):
                r[(i,j)]=zeros(len(self.alg.basis),1)

        if not (2 in c.vd): return r
        for w in c.vd[2]:
            v=c.vd[2][w]
            for t in range(len(c.vd[2][w])):
                if v[t]!=0:
                    i,j,k=[self.alg.basis.index(A) for A in self.basis(2,w)[t].components]
                    r[(i,j)][k]=v[t]
        return r       
    
    def one_cochain_mat_rep(self,c):
        """Returns a matrix representation of c as an element of Hom(g,g)
        INPUTS:
        * 'c' - a 1 cochain from self
        """
        if 1 not in c.vd: return zeros(len(self.alg.basis))
        r=zeros(len(self.alg.basis))
        for w in c.vd[1]:
            for a in range(len(c.vd[1][w])):
                # To do
                i,j=[self.alg.basis.index(A) for A in self.basis(1,w)[a].components]
                r[j,i]=c.vd[1][w][a]
        return r
    # def harm_proj(self,c):
    #     """Returns the orthogonal projection of c onto the harmonic subspace
    #     INPUTS:
    #     * 'c' - a cochain from self
    #     """
    #     return c.harm_proj()

The above (right) $GL_+(\mathfrak{g})$ action on $C^2(\mathfrak{m},\mathfrak{g})$ is given by 

$$A.c(u,v) = A^{-1}\cdot c(Au,Av)\quad \text{or}\quad A.(X_1^*\wedge X_2^*\otimes X_3) = (A^TX_1^*)\wedge(A^TX_2^*)\otimes A^{-1}X_3,$$

which is the naturally induced action. Representing $c$ as a matrix $M_c$, we can also write

$$ M_{A. c} = A^{-1}\cdot c\cdot (A\wedge A)|_{\mathfrak{m}\wedge \mathfrak{m}}$$

In [ ]:
class ext_alg(Tensor_alg):
    def __init__(self,T_symb_obj):
        T_symb_obj.ext_alg=self
        Tensor_alg.__init__(self,T_symb_obj)
        self.childcls=ext_elt
    
    def wedge_tuples(self,tuple1,tuple2,obj2_type):
        '''Returns (wedge,sgn), where wedge is a tuple representing 
        tuple1 wedge tuple2 and sgn is -1 or 1
        
        INPUTS:
        * 'tuple1', 'tuple2' - tuples of T_symb_basis_elt objects
        * 'obj2_type' - among 'ext_elt' and 'cochain', indicating the type of 
                        the object repped by tuple2'''
        if obj2_type not in ['ext_elt','cochain']: raise invalid_parent_exception('wedge_tuples recieved invalid parent type as arg')
        if obj2_type == 'ext_elt': B=tuple2
        else: B=tuple(list(tuple2)[0:-1])
        #check for repeats
        if len(set(tuple1).union(set(B)))!=len(tuple1)+len(B):
            return 'Nil'
        t,s=sort_basis_tuple(tuple1+B,self.alg.basis_strs)
        if obj2_type=='ext_elt': return (t,s)
        return (t+(tuple2[-1],),s)
    
    def tuple_wght(self,t):
        ''' Returns the weight of t
        INPUTS:
        * 't' - a tuple representing an element of self
        '''
        r=0
        for A in t:
            i=self.alg.basis_strs.index(A)
            r=r-self.alg.wght_list[i] # This is the exterior algebra of m_dual
        return r
        
    def tuple_deg(self,t):
        ''' Returns the degree of t
        INPUTS:
        * 't' - a tuple representing an element of self
        '''
        return len(t)
    
        
    def dwi(self,basis_str_tuple):
        """Returns the degree, weight, and index of basis_tuple
        Inputs:
        * 'basis_tuple' -- a tuple of basis strings representing a basic cochain
        """
        if basis_str_tuple not in self.dwi_dicts: self.init_basis(len(basis_str_tuple))
        return self.dwi_dicts[basis_str_tuple]
    
    def sort_tuple(self,t):
        """Returns an (ext alg) sorting of t and the sign of the corresponding permutation, as a tuple
        or None if t contains repeats
        INPUTS:
        * 't' - a tuple of basis_strs
        """
        if len(t)!=len(set(t)): return (None,0)
        return(sort_basis_tuple(t,self.alg.basis_strs))
            
    def init_basis(self,deg):
        """Sets value of deg in basis_cache and adds to basis_dicts"""
        if deg in self.basis_cache: return None
        deg_subsets=[A for A in list(combinations([str(A) for A in self.alg.m_basis],deg))]
        self.basis_cache[deg]={}
        wght_ct={}
        for i in range(len(deg_subsets)):
            # count the number of elements of deg d and wght w
            # and set the deg, wght, and index of each elt in dwi_dicts
            A=deg_subsets[i]
            w=self.tuple_wght(A)
            if not w in wght_ct: 
                j=0
                wght_ct[w]=1
            else: 
                j=wght_ct[w]
                wght_ct[w]+=1
            self.dwi_dicts[A]=(deg,w,j)
        for A in deg_subsets:
            d,w,i=self.dwi_dicts[A]
            vec=SparseMatrix(zeros(wght_ct[w],1))
            vec[i]=1
            vd={d:{w:vec}}
            b_elt=ext_basis_elt(self,d,w,vd,A)
            if i==0:self.basis_cache[deg][w]=[b_elt]
            else: self.basis_cache[deg][w].append(b_elt)
            
    def wedge_indices(self,d1,w1,i1,d2,w2,i2,obj2_type):
        """Returns a tuple with deg, wght, index, and sign of the tuples
        or ('Nil','Nil','Nil',0) if the wedge is zero
        INPUTS:
        * 'd1','d2' - degrees
        * 'w1','w2' - weights
        * 'i1','i2' - indices
        * 'obj2_type' - either 'ext_elt' or 'cochain', giving the type of 
                        the object specified by (d2,w2,i2)
        """
        if obj2_type not in ['ext_elt','cochain']: raise invalid_parent_exception('wedge_indices recieved invalid parent as arg')
        
        t1=tuple([str(A) for A in self.basis(d1,w1)[i1].components])
        if obj2_type=='ext_elt': t2=tuple([str(A) for A in self.basis(d2,w2)[i2].components])
        else: t2=tuple([str(A) for A in self.alg.cochain_complex.basis(d2,w2)[i2].components])
        
        r=self.wedge_tuples(t1,t2,obj2_type)
        if r=='Nil': return ('Nil','Nil','Nil',0)
        t3,s=(r[0],r[1]) # resulting tuple and sign
        if obj2_type=='ext_elt':i3=self.basis_strs(d1+d2,w1+w2).index(t3)
        else: i3=self.alg.cochain_complex.basis_strs(d1+d2,w1+w2).index(t3)
        return (d1+d2,w1+w2,i3,s) 

In [ ]:
class cochain(Tensor_alg_elt):
    def __init__(self,parent,vd={}):
        Tensor_alg_elt.__init__(self,parent,vd)
        
    @classmethod
    def from_cd(cls,parent,cd={}):
        """Constructs an exterior element in parent from a coefficient dictionary"""
        return parent.elt(parent.cd_to_vd(cd))
    
    def __gt__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)>str(other)
    
    def __ge__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)>=str(other)
    
    def __lt__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)<str(other)
    
    def __le__(self,other):
        if self.parent!=other.parent: raise invalid_parent_exception
        return str(self)<=str(other)

    def symb_expr(self):
        expr=0
        for d in self.vd:
            for w in self.vd[d]:
                for i in range(len(self.vd[d][w])):
                    if self.vd[d][w][i]!=0: expr+=self.vd[d][w][i]*self.parent.basis(d,w)[i].symb
        return expr

    def pprint(self):
        expr=0
        for d in self.vd:
            for w in self.vd[d]:
                for i in range(len(self.vd[d][w])):
                    if self.vd[d][w][i]!=0: expr+=self.vd[d][w][i]*self.parent.basis(d,w)[i].symb
        display(simplify(expr))
    
    def cb(self):
        """Returns the coboundary map of C(m,g) applied to c
           """
        r={}
        for d in self.vd:
            for w in self.vd[d]:
                if shape(self.parent.cb_mat(d,w))[1]==0: v=zeros(len(self.parent.basis(d+1,w)),1)
                else: v=self.parent.cb_mat(d,w)*self.vd[d][w]
                if d+1 not in r: r[d+1]={}
                if w not in r[d+1]: r[d+1][w]=SparseMatrix(zeros(len(self.parent.basis(d+1,w)),1))
                r[d+1][w]=r[d+1][w]+v
        return self.parent.elt(r) 
    
    def cb_preim_elt(self):
        """Returns a cochain which maps to self under the coboundary.
        If self is not exact, returns None.
        """
        return self.parent.cb_preim_elt(self)
    
    def check_valid_vd(self):
        """Returns True if self has a vector dictionary which is a 
        valid representation of a cochain from self.parent, False otherwise
        """
        for d in self.vd:
            for w in self.vd[d]:
                if shape(self.vd[d][w])!=(len(self.parent.basis(d,w)),1): return False
        return True

    def wght_proj(self,w):
        """returns: a cochain representing the projection of self onto weight w
        INPUTS:
        * 'w' -- a positive integer weight
        """
        nvd={d:{w:self.vd[d][w]} for d in self.vd}
        return self.parent.elt(nvd)
    
    # def harm_proj(self):
    #     '''returns: The orthogonal projection (w.r.t iprod) of c onto the harmonic subspace of C(g_-,g)'''
    #     return self.parent.harm_proj(self)

In [ ]:
class ext_elt(Tensor_alg_elt):
    def __init__(self,parent,vd={}):
        Tensor_alg_elt.__init__(self,parent,vd)
        
    @classmethod
    def from_cd(cls,parent,cd={}):
        """Constructs an exterior element in parent from a coefficient dictionary"""
        return parent.elt(parent.cd_to_vd(cd))

    def wedge(self,other):
        """ Returns the wedge product of self and other
        INPUTS:
        * 'other' - another exterior element or a cochain
        
        NOTE: Since it's ambiguous whether an algebra element is dual or not, 
              other cannot be of type T_symb_elt
        """
        
        obj2_type=None        
        if other.parent==self.parent:
            obj2_type='ext_elt'
            r=self.parent.elt({})
        if other.parent==self.parent.alg.cochain_complex:
            obj2_type='cochain'
            r=self.parent.alg.cochain_complex.elt({})
        if obj2_type==None: raise invalid_parent_exception('wedge recieved arguments with incompatible parents')
        
        L1=[]
        for d1 in self.vd:
            for w1 in self.vd[d1]:
                for i1 in range(len(self.vd[d1][w1])):
                    if self.vd[d1][w1][i1]!=0: L1.append((d1,w1,i1,self.vd[d1][w1][i1]))
        L2=[]
        for d2 in other.vd:
            for w2 in other.vd[d2]:
                for i2 in range(len(other.vd[d2][w2])):
                    if other.vd[d2][w2][i2]!=0: L2.append((d2,w2,i2,other.vd[d2][w2][i2]))
        for t1 in L1:
            d1,w1,i1,c1=t1
            for t2 in L2:
                d2,w2,i2,c2=t2
                d3,w3,i3,s=self.parent.wedge_indices(d1,w1,i1,d2,w2,i2,obj2_type)
                if s!=0: r.update_add(d3,w3,i3,s*c1*c2)
        return r

In [ ]:
class Tensor_alg_basis_elt(Tensor_alg_elt):
    def __init__(self,parent,deg,wght,vd,str_rep):
        self.deg=deg
        self.wght=wght
        self.str_rep=str_rep
        Tensor_alg_elt.__init__(self,parent,vd)
        alg=parent.alg
        self.components=[alg.basis[alg.basis_strs.index(A)] for A in str_rep]
    
    def __str__(self):
        return tuple_to_str(self.str_rep)
    
    def __repr__(self):
        return self.__str__()

In [ ]:
class ext_basis_elt(ext_elt,Tensor_alg_basis_elt):
    def __init__(self,parent,deg,wght,vd,str_rep):
        ext_elt.__init__(self,parent,vd)
        Tensor_alg_basis_elt.__init__(self,parent,deg,wght,vd,str_rep)
        self.length=None
        if parent.alg.Q!=None:
            self.length=Rational(1,prod([A.length for A in self.components]))

For a basis element $\phi = X_1^*\wedge\cdots\wedge X_k^*\otimes Y$, we have
$$
    \partial\phi = \sum_{a\not\in\{1,\ldots,k\}} X_a^*\wedge X_1^*\wedge\cdots \wedge X_k^*\otimes[X_a,Y]
    \\
    +\sum_{a<b\not\in\{1,\ldots, k\}}\sum_{i\in \{1,\ldots, k\}}(-1)^i\Big(X_i^* [X_a,X_b]\Big)X_a^*\wedge X_b^*\wedge X_1^*\wedge\cdots\wedge\widehat {X_i^*}\wedge\cdots X_k^*\otimes Y
$$

In [ ]:
class cochain_basis_elt(cochain,Tensor_alg_basis_elt):
    def __init__(self,parent,deg,wght,vd,str_rep):
        cochain.__init__(self,parent,vd)
        Tensor_alg_basis_elt.__init__(self,parent,deg,wght,vd,str_rep)
        self.length=None
        if parent.alg.Q!=None:
            self.length=Rational(self.components[-1].length,
                                 prod([A.length for A in self.components[0:-1]]))
            
        # Set the symbol representation of self
        sl=[str(A) for A in self.components]
        s='{'*(len(sl)-1)
        for i in range(len(sl)-2):
            s+=sl[i]
            s+='^*\\wedge}'
        s+=sl[len(sl)-2]+'^*\\otimes}'+sl[-1]
        self.symb=symbols(s)
            
    def cb_vec(self):
        """Returns the vector representing coboundary(self) in the basis C^{deg+1}_{wght}(m,g)
        Should only be called by cochain.cb_mat.
        """
        mb=self.parent.alg.m_basis
        Y=self.components[-1]
        r=[0]*len(self.parent.basis(self.deg+1,self.wght))
        
        B_set=set([mb.index(A) for A in self.components[0:-1]])
        non_B_set=set(range(len(mb))).difference(B_set)
        
        # First term
        for a in non_B_set:
            Xa=mb[a]
            # compute the RHS above, im_a_vec: [X_a,Y]+sum_{i} X_i^*[X_a,X_i]Y
            im_a_vec=Xa.ad(Y).vec
            for j in range(len(im_a_vec)):
                if im_a_vec[j]!=0:
                    tl=[Xa]+self.components[0:-1]+[self.parent.alg.basis[j]]
                    t,s=self.parent.sort_tuple(tuple([str(A) for A in tl]))
                    if s!=0:
                        ind=self.parent.basis_strs(self.deg+1,self.wght).index(t)
                        r[ind]+=s*im_a_vec[j]
        # Second term
        for a in non_B_set:
            for b in non_B_set:
                if a<b:
                    Xa=mb[a]
                    Xab=Xa.ad_mat(mod='m').col(b)
                    for i in B_set:
                        if Xab[i]!=0:
                            cpts=copy.copy(self.components[0:-1])
                            cpts.remove(mb[i])
                            tl=[Xa,mb[b]]+cpts+[self.components[-1]]
                            t,s=self.parent.sort_tuple(tuple([str(A) for A in tl]))
                            if s!=0:
                                ind=self.parent.basis_strs(self.deg+1,self.wght).index(t)
                                r[ind]+=(-1)**(1+self.components.index(mb[i]))*s*Xab[i]
        
        return SparseMatrix([r]).transpose()

In [ ]:
class cochain_init_Exception(Exception):
     def __init__(self, message=""):
        self.message = message
        super().__init__(self.message)

In [ ]:
class nonhomogeneous_Exception(Exception):
    def __init__(self,message=""):
        self.message = message
        super().__init__(self.message)

If the curvature $K_{<d}$ is known, then we compute $K_d$ as follows:

$$K_{\geq d}(v_1,v_1) = F^{-1}\Big([F(v_1),F(v_2)]-F[v_1,v_2]-F\circ K_{< d}(v_1,v_2)\Big)$$

## Deprecated

In [ ]:
# def SF_ad(X1,X2,SF):
#     '''args: X1,X2 are VFs, either T_elts or vectors/lists with coeffs indexed objects and 
#              the coordinates y,h,e SF is the structure function defining the ad-relations between 
#              T basis elts, an element of C, the cochain complex
#        returns: [X1,X2], where the str function defines the relations between T basis elts,
#                and K,y,h,e depend on the coordinates appropriately'''
#     T=SF.parent.alg
#     result=0
#     X=[X1,X2]
#     v=[None,None]
#     for j in range(2):
#         Xj=X[j]
#         vj=v[j]
#         if type(Xj)==list: v[j]=Xj
#         if type(Xj)==type(eye(4)): v[j]=list(Xj)
#         if type(Xj)==T_symb_elt or type(Xj)==T_symb_basis_elt: v[j]=Xj.vec_rep
#     for i in range(len(v[0])):
#         coeff_1=v[0][i]
#         if coeff_1!=0:
#             for j in range(len(v[1])):
#                 coeff_2=v[1][j]
#                 if coeff_2!=0:
#                     res1=coeff_1*abn_ind_der(coeff_2,i)*T.basis[j]
#                     res2=-coeff_2*abn_ind_der(coeff_1,j)*T.basis[i]
#                     new_res=result+res1+res2
#                     result+=res1
#                     result+=res2
#     e_elt=T.elt(v[0]).cast_as_ext_elt().wedge(T.elt(v[1]).cast_as_ext_elt())
#     result+=SF.apply_cochain_map(e_elt)
#     return result

In [ ]:
# T=T_symb(5)
# C=T.cochain_complex
# c=C.cochain({('e1','e2','e3','Y'):1})
# c.find_mat_rep()
# d=C.cochain(c.mat_rep,deg=3,mat_rep=True)

In [ ]:
# coeff_dict=c.mat_rep
# parent=C
# deg=3
# wght="UNKNOWN"
# mat_rep=True

# d.parent=parent
# d.deg=deg
# d.wght=wght
# d.heis_dim=parent.heis_dim

# if mat_rep and coeff_dict!='Nil':
#     if d.deg=='UNKNOWN': 
#         raise cochain_init_Exception('If mat_rep is specified, deg must be specified')
#     print('check 1')
#     d.parent.ext_alg.init_basis(deg)
#     d.parent.init_basis(deg)
#     d.mat_rep=coeff_dict
#     d.coeff_dict={}
#     pprint(coeff_dict)

#     m=len(d.parent.alg.basis)
#     n=len(d.parent.ext_alg.basis[deg]) # mat_rep is a map Rn-->Rm
#     if shape(d.mat_rep)!=(m,n):
#         raise cochain_init_Exception('Incompatible mat_rep for deg ='+str(deg))
#     for j in range(d.mat_rep.cols):
#         col = d.mat_rep.col(j)
#         print('col',j,'=',col)
#         for k in range(len(col)):
#             J=d.parent.ext_alg.basis_strs[deg][j]
#             ind=tuple(list(J[0:d.deg])+[d.parent.alg.basis_strs[k]])
#             d.coeff_dict[ind]=d.mat_rep[k,j]
# else:
#     if coeff_dict=='Nil': coeff_dict={}
#     temp=remove_zeros({parent.sort_cochain_tuple(A)[0]:
#                                   parent.sort_cochain_tuple(A)[1]*coeff_dict[A] for A in coeff_dict})
#     d.coeff_dict=remove_antisymm_zeros_cochains(temp)
#     d.mat_rep=None

In [ ]:
# '''Computes the matrix rep of a cochain in C(m,g), (NOT in C(g,g))'''
# if c==c.parent.cochain():
#     c.mat_rep='Nil'
#     print('Nil')

# C=c.parent
# E=C.ext_alg
# T=C.alg
# d=C.deg(c)

# if d=='Nil':
#     print('Nil')

# E.init_basis(d)
# C.init_basis(d)
# result=zeros(len(T.basis),len(E.basis_lists[d]))
# for key in c.coeff_dict:
#     try:
#         j=E.basis_indices[d][tuple(key[0:d])]
#         i=T.basis_strs.index(key[d])
#         result[i,j]=c.coeff_dict[key]
#     except(KeyError):
#         if not key[0] in T.basis_strs[3:len(T.basis)]: print('find_mat_rep error:', key[0],'not in m')
#         elif not key[0] in T.basis_strs[3:len(T.basis)]: print('find_mat_rep error:', key[0],'not in m')
#         else: pass
# c.mat_rep=result

In [ ]:
# result

In [ ]:
# def curv_func(T,fm,sf,d,prev_K=None):
#     '''arg: T, a T_symb object
#             fm, a matrix whose columns represent an algebraic frame in the basis of T_symb;
#                 that is, it is block unipotent upper triangular
#             sf, the structure function or curvature for the basis in which fm is written
#             d, a degree
#             prev_K: the curvature function of the degree (d-1) (not the structure function)
#        returns: A cochain representing the curvature of fm deg 1 to degree d'''
#     C=T.cochain_complex
#     r_dict={}
    
#     if prev_K==None:
#         for i in range(len(T.basis)):
#             A=T.elt(fm.col(i))
#             for j in range(i,len(T.basis)):
#                 B=T.elt(fm.col(j))
#                 w=Matrix(SF_ad(A,B,sf).vec_rep)

#                 # # How many of these do we need to check?
#                 # # from wght wght(i)+wght(j) to wght(i)+wght(j)+d
#                 # # Lower ones will be zero automatically
#                 min_w=T.wght_list[i]+T.wght_list[j]
#                 max_w=min(min_w+d,T.wght_list[0])

#                 for k in reversed(range(len(T.basis))):
#                     if T.wght_list[k]>=min_w and T.wght_list[k]<=max_w:
#                         if fm[k,k]!=0: # This frame matrix should be unipotent, really
#                             coeff=w[k]/fm[k,k]
#                             w+=Matrix([-coeff*A for A in fm.col(k)])
#                             # Don't add the degree zero part to the result
#                             if T.wght_list[k]>min_w and coeff!=0:
#                                 r_dict[(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])]=coeff
#         return C.elt_from_cd(r_dict)

#     for i in range(len(T.basis)):
#         A=T.elt(fm.col(i))
#         for j in range(i,len(T.basis)):
#             B=T.elt(fm.col(j))
#             w=Matrix(SF_ad(A,B,sf).vec_rep)

#             # Remove the degree zero part
#             for k in reversed(range(len(T.basis))):
#                 if T.wght_list[k]==T.wght_list[i]+T.wght_list[j]:
#                     coeff=w[k] # If the frame were not unipotent, we would need /fm[k,k] here
#                     w+=Matrix([-coeff*A for A in fm.col(k)])
            
#             # Remove degree btwn 1 and (d-1)
#             ev=T.basis[i].cast_as_ext_elt().wedge(T.basis[j])
#             w+=-fm*Matrix(prev_K.apply_cochain_map(ev).vec_rep)
                
#             # Compute the degree d part
#             for k in reversed(range(len(T.basis))):      
#                 if T.wght_list[k]==T.wght_list[i]+T.wght_list[j]+d:
#                     if fm[k,k]!=0: # This frame matrix should be unipotent, really
#                         coeff=w[k]/fm[k,k]
#                         w+=Matrix([-coeff*A for A in fm.col(k)])
#                         if coeff!=0: r_dict[(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])]=coeff
#     return C.elt_from_cd(r_dict)+prev_K

In [ ]:
# # # For time control purposes
# def alt_curv_func(T,fm,sf,d,prev_K=None):
#     '''arg: T, a T_symb object
#             fm, a matrix whose columns represent an algebraic frame in the basis of T_symb;
#                 that is, it is block unipotent upper triangular
#             sf, the structure function or curvature for the basis in which fm is written
#             d, a degree
#             prev_K: the curvature function of the degree (d-1) (not the structure function)
#        returns: A cochain representing the curvature of fm deg 1 to degree d'''
#     C=T.cochain_complex
#     r_dict={}
    
#     if prev_K==None:
#         for i in range(len(T.basis)):
#             A=T.elt(fm.col(i))
#             for j in range(i,len(T.basis)):
#                 B=T.elt(fm.col(j))
#                 w=Matrix(SF_ad(A,B,sf).vec_rep)

#                 # # How many of these do we need to check?
#                 # # from wght wght(i)+wght(j) to wght(i)+wght(j)+d
#                 # # Lower ones will be zero automatically
#                 min_w=T.wght_list[i]+T.wght_list[j]
#                 max_w=min(min_w+d,T.wght_list[0])

#                 for k in reversed(range(len(T.basis))):
#                     if T.wght_list[k]>=min_w and T.wght_list[k]<=max_w:
#                         if fm[k,k]!=0: # This frame matrix should be unipotent, really
#                             coeff=w[k]/fm[k,k]
#                             w+=Matrix([-coeff*A for A in fm.col(k)])
#                             # Don't add the degree zero part to the result
#                             if T.wght_list[k]>min_w and coeff!=0:
#                                 r_dict[(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])]=coeff
#         return C.cochain(r_dict)

#     for i in range(len(T.basis)):
#         A=T.elt(fm.col(i))
#         for j in range(i,len(T.basis)):
#             B=T.elt(fm.col(j))
#             w=Matrix(SF_ad(A,B,sf).vec_rep)

#             # Remove the degree zero part
#             for k in reversed(range(len(T.basis))):
#                 if T.wght_list[k]==T.wght_list[i]+T.wght_list[j]:
#                     if fm[k,k]!=0: # This frame matrix should be unipotent, really
#                         coeff=w[k]/fm[k,k]
#                         w+=Matrix([-coeff*A for A in fm.col(k)])
            
            
#             # Remove degree btwn 1 and (d-1)
#             ev=T.basis[i].cast_as_ext_elt().wedge(T.basis[j])
#             w+=-fm*Matrix(prev_K.apply_cochain_map(ev).vec_rep)
                
#             # Compute the degree d part
#             for k in reversed(range(len(T.basis))):      
#                 if T.wght_list[k]==T.wght_list[i]+T.wght_list[j]+d:
#                     if fm[k,k]!=0: # This frame matrix should be unipotent, really
#                         coeff=w[k]/fm[k,k]
#                         w+=Matrix([-coeff*A for A in fm.col(k)])
#                         if coeff!=0: r_dict[(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])]=coeff
#     return C.cochain(r_dict)+prev_K

In [ ]:
# def str_func(T,fm,sf,wl):
#     '''arg: T, a T_symb object
#             fm, a matrix whose columns represent an algebraic frame in the basis of T_symb;
#                 that is, it is block upper triangular (with non)
#             sf, the structure function for the basis
#             wl, a list of wghts
#        returns: a list of str funcs for the frame represented by fm, corresponding to wghts in wl'''
#     r_dict={}
#     fm_inv=fm.inv()
#     for i in range(len(T.basis)):
#         A_vec=fm.col(i)
#         A=T.elt(A_vec)
#         for j in range(i,len(T.basis)):
#             B_vec=fm.col(j)
#             B=T.elt(B_vec)
#             w=fm_inv*Matrix(SF_ad(A,B,sf).vec_rep) # coords of ad(A)(B)
#             for k in range(len(T.basis)):
#                 r_dict[(T.basis_strs[i],T.basis_strs[j],T.basis_strs[k])]=w[k]
#     return T.cochain_complex.cochain(r_dict)

In [ ]:
# # An ad hoc test of coboundary_preim
# T=T_symb(9)
# C=T.cochain_complex
# C.set_coker(2)
# print(C.coboundary_preim.keys()==C.coboundary_im.keys())
# for deg in C.coboundary_preim:
#     for wght in C.coboundary_preim[deg]:
#         test1=[A.coboundary() for A in C.coboundary_preim[deg][wght]]
#         test2=C.coboundary_im[deg][wght]
#         if test1!=test2:
#             print('Failure at deg =',deg,'wght =',wght,'\n')
#             print(test1,'\n')
#             print(test2)

In [ ]:
# T3=T_symb(3)
# T5=T_symb(5)
# T7=T_symb(7)
# T9=T_symb(9)

# C3=T3.cochain_complex
# C5=T5.cochain_complex
# C7=T7.cochain_complex
# C9=T9.cochain_complex

In [ ]:
# C5.set_harm_basis(2)
# C7.set_harm_basis(2)

In [ ]:
# C9.set_harm_basis(2)

In [ ]:
# C7.set_coker(2)

In [ ]:
# for A in C5.harm_basis[2]:
#     print(A,':',len(C7.harm_basis[2][A]))


In [ ]:
# for A in C7.harm_basis[2]:
#     print(A,':',len(C7.harm_basis[2][A]))


In [ ]:
# for A in C9.harm_basis[2]:
#     print(A,':',len(C9.harm_basis[2][A]))


In [ ]:
#C7.set_coker(2)
#C9.set_coker(2)

In [ ]:
#C3.coker[2]
#C3.coboundary_im[2]
#C3.harm_basis[2]

In [ ]:
#C5.coker[2]
#C5.coboundary_im[2]
#C5.harm_basis[2]

In [ ]:
#C7.coker[2]
#C7.coboundary_im[2]
#C7.harm_basis[2]

In [ ]:
#C9.coker[2]
#C9.coboundary_im[2]
#C9.harm_basis[2]